In [ ]:
import fiona
import geopandas as gpd

import os
from pathlib import Path
import tempfile
import zipfile

In [ ]:
# Folder with WIA in Sharepoint
base_dir = "./"
data_in_folder = base_dir + "data_in/"
data_out_folder = base_dir + "data_out/templates/"

# shapes subfolder
data_in_shapes = data_in_folder + "geo_sight/"

country_zip_shapes = [
    "global_admin_boundaries_matched_latest.gdb.zip",
    # "global_admin_boundaries_matched_historic.gdb.zip"
]

# admin level selected
adm_level = 2

# column pcode and name (assume harmonized across CODs)
pcode_col = f"adm{adm_level}_pcode"
name_col = f"adm{adm_level}_name"
parent_pcode_col = f"adm{adm_level-1}_pcode"


In [ ]:
# Extract the country shape zip file into a temporary directory
with tempfile.TemporaryDirectory() as temp_dir:

    temp_dir = Path(temp_dir)

    # Create a folder with the .gdb extension
    gdb_path = temp_dir / "data.gdb"
    gdb_path.mkdir()

    # Extract all files into the .gdb folder
    with zipfile.ZipFile(
        data_in_shapes + country_zip_shapes[0],
        'r',
    ) as zip_file:
        zip_file.extractall(gdb_path)

    # check layer if gdb
    adm_layer = [
        l for l in fiona.listlayers(gdb_path)
    ]

    # read country_admin_shape as geodataframe
    gdf = gpd.read_file(
        gdb_path,
        layer = adm_layer[adm_level - 1],
    ).to_crs('EPSG:4326')


In [ ]:
# Extract the country shape zip file into a temporary directory
with tempfile.TemporaryDirectory() as temp_dir:

    temp_dir = Path(temp_dir)

    # Create a folder with the .gdb extension
    gdb_path = temp_dir / "data.gdb"
    gdb_path.mkdir()

    # Extract all files into the .gdb folder
    with zipfile.ZipFile(
        data_in_shapes + "global_admin_boundaries_matched_historic.gdb.zip",
        'r',
    ) as zip_file:
        zip_file.extractall(gdb_path)

    # check layer if gdb
    adm_layer = [
        l for l in fiona.listlayers(gdb_path)
    ]

    # read country_admin_shape as geodataframe
    gdf_hist = gpd.read_file(
        gdb_path,
        layer = adm_layer[adm_level - 1],
    ).to_crs('EPSG:4326')


In [ ]:
# gdf.query("adm0_name == 'Somalia'").query("adm1_name == 'Banadir'").shape
# gdf.query("adm0_name == 'Somalia'").version
# 91-18
# gdf.query("adm0_name == 'Lebanon'").version
# gdf.query("iso3 == 'COD'")
# gdf.query("iso3 == 'COL'")
# gdf.query("iso3 == 'MMR'")
# gdf.query("adm3_pcode.str.contains('MMR017')")
# gdf.query("iso3 == 'MMR'")[
#     gdf.adm3_pcode == gdf.adm2_pcode
# ]
# gdf.query("adm0_name == 'Myanmar'")[
#     [
#         "adm2_name",
#         "adm2_pcode",
#         "adm1_name",
#         "adm1_pcode",
#         "adm0_name",
#         "adm0_pcode",
#         "lang",
#         "iso2",
#         "iso3",
#         "version",
#         "valid_on",
#         "valid_to",
#         "adm_origin"
#     ]
# ]
gdf.query("iso3 == 'MOZ'")

In [ ]:
# gdf_hist.adm0_name.unique()
# gdf_hist.query("adm0_name == 'Colombia'")[
# gdf.query("adm0_name == 'Colombia'")[
#     [
#         "adm2_name",
#         "adm2_pcode",
#         "adm1_name",
#         "adm1_pcode",
#         "adm0_name",
#         "adm0_pcode",
#         "lang",
#         "iso2",
#         "iso3",
#         "version",
#         "valid_on",
#         "valid_to",
#         "adm_origin"
#     ]
# ]
gdf_hist.query("iso3 == 'MOZ' and adm1_pcode == 'MZ08'").sort_values(by="adm2_pcode")

In [ ]:
gdf.query("adm0_name == 'Somalia'").query("version == 'v02'").version

In [ ]:
# gdf.query("adm0_name == 'Afghanistan'").query("adm1_name == 'Samangan'").sort_values(by="adm2_pcode")
gdf.query("adm0_name == 'Cameroon'").sort_values(by="adm2_pcode")

In [ ]:
# shapefile output file name
shape_filename = f"cod-ab-global_edge-matched_{adm_layer[0]}"

# TODO: add constant source column: "Multiple sources aggregated by UN OCHA Information Systems Section (ISS)"

# export zip file with shapefile (ESRI Shapefile needs multiple sidecar files)
with tempfile.TemporaryDirectory() as temp_shp_dir:
    shp_path = os.path.join(temp_shp_dir, f"{shape_filename}.shp")
    gdf[
        [
            pcode_col,
            parent_pcode_col,
            name_col,
            "lang",
            "iso3",
            "version",
            "geometry",
        ]
    ].rename(
        columns={
            pcode_col: "Pcode",
            parent_pcode_col: "Parent_Pcode",
            name_col: "Name",
            "lang": "Language",
            "iso3": "ISO3",
            "version": "Version",
        }
    ).to_file(shp_path, driver="ESRI Shapefile")

    with zipfile.ZipFile(
        data_in_shapes + f"{shape_filename}.zip",
        'w',
        zipfile.ZIP_DEFLATED,
    ) as zip_file:
        for f in os.listdir(temp_shp_dir):
            zip_file.write(os.path.join(temp_shp_dir, f), arcname=f)
